__Kubeflow__ - инструмент трекинга ML-экспериментов (ML toolkit), имеющий плотную интегрирацию с Kubenetes<br>Это аналог ML-FLOW

Какая связь с  Kubernetes - Kubeflow контейнеризуют отдельные блоки ML-пайплайнов, которые разворачиваются на Kubernetes кластере. То есть это не только трекинг, но и управление инфраструктурой в одном флаконе

__Почему ML on Containers?__<br>
Главный плюс запуска ML-приложения в контейнере тот же, как для обычного приложения - легкость конфигурации, а именно:
- __масштабируемость__<br>требуемые ресурсы и инстансы параметризованы
- __консистентность__<br>версии внутри контейнера согласованы
- __портируемость__<br>не нужна настройка под среду
- __легковесные__<br>настраивается только целевая часть, инфраструктура выносится за скобки

__Почему ML on Kubernetes__<br>
- Reusability - можно переипользовать
- Portability - можно легко перемещать облако / локальный кластер
- Scaling - Kubernetes сам отвечает за 

Чтобы не ставить библиотеки с нуля и не настраивать свои контейнеры, в облаках (например AWS) есть репозитории Docker Deep Learning обьразов

Если в разработке два основных цикла это Development и Deployment, в ML проектах два основных цикла это Training и Serving. И в Kubeflow есть компоненты, поддерживающие эту функциональность

<img src="img/kubeflow1.svg" width=750>

__Компоненты Kubeflow__<br>
Физически, Kubeflow - еще один процесс на Kubernetes кластере

Компонент Kubeflow Notebooks - это spawner Jupyter ноутбуков (серверов) для аналитиков. По сути это их вариант JupyterHub

Кроме того там есть куча кастомных инструментов (они называют это проектами), которые можно использовать в ML пайплайне. Почему - видимо, потому что Google, почему нет:
- Katib<br>оптимизатор гиперпараметров, циклическике запуски
- Kale<br>
- KServe<br>сервер ML-моделей
- Minio<br>свой S3 сторадж
- KServe<br>

Так же как с Minikube, для Kubeflow есть 1-node приложение __MiniKF__, на котором можно учиться. Контейнер можно найти в репозиториях Docker образов, например, в GCP














### Kubeflow Pipelines
For Experiments, Jobs, Runs  
Pipeline = DAG из выполняемых шагов. Каждый компонент = контейнер

Собрать компонент можно из преднастроенного класса, например ContainerOp. Он собирает контейнер из выбранного Docker образа, выполняет команды на нем и .  Python decorator

Pipeline = произвольная функция, в которой прописывается логика вызова копонентов. Её нужно пометить декоратором и можно компилировать

Компиляция<br>Compile => генерирует чистый YAML configuration file, который можно запускать на любом Kubernetes кластере<br>Kubeflow собирает zip файл почему-то

Далее хотим запустить пайплайн на кластере:
1) через UI upload zip файла
2) через SDK, создаем эксперимент и делаем run

Чтобы это запустилось, нужно подключить
`import kfp`<br>
`import kfp.dsl` для декораторов и операторов

Кроме того контейнер можно собрать из 



## Images
<img src="img/kubeflow_images.png" width=750>

## KServe

KServe - это Kubeflow компонент для сервинга моделей. Аналог движков сервинга общего назначения torchserve / tensorserve / mlserve / tritonserver или генеративных типа vLLM / ollama

Reminder, зачем вообще нужен движок сервинга моделей
1) открыть модель миру - обернуть model.predict() в API сервис
2) авто-скейлить модель под нагрузку
3) встроенные возможности по A/B тестированию: балансировать между версиями
4) аутентификация пользователей и безопасность
5) мониторинг и логирование запросов
6) версионирование моделей
7) оптимизация выполнения запросов
8) можно собирать сложный мульти-языковой pipeline обработки (pre-process, predict, post-process)
9) поддержка доступности: warmup, graceful shutdown etc

Что включается в оптимизация выполнения запросов
- объединение в единый батч
- асинхронное выполнение запроса (с передачей выполнения)
- совместное использование GPU-ресурсов
- маршрутизация между моделями

Что такое поддержка достцупности?
- Warmup<br>в runtime-ах много оптимизации делается при первом фактическом запросе - чтобы запросы не тупили, делают "прогрев" модели
- Graceful shutdown<br>если запускается shutdown пода (при обновлении версии), надо корректно завершить уже бегущие запросы
- Readiness probe<br>под становится доступным в балансировщике только когда все веса подгружены и 
- Healthcheck<br>нужно периодически проверять статус моделей в подах
- Hooks<br>кастомные команды, срабатывающие при опредедленных тригеррах

<img src="img/kserve.svg" width=750>